# 09 — ResNet-18 fine-tune with LOSO-CV

Transfer-learning baseline. `src.models.resnet18_finetune` adapts conv1 to
2 input channels and re-heads the final fc layer.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.io as sio

plt.rcParams["figure.dpi"] = 110
plt.rcParams["figure.figsize"] = (10, 4)

import torch

from src.models import resnet18_finetune, count_parameters
from src.train import loso_cv, loso_summary, train_one_fold
from src.dataset import WindowDataset


## 1. Build the model and inspect parameter count

In [ ]:
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"device: {device}")

model = resnet18_finetune(in_channels=2, pretrained=True, freeze_until="layer3")
print(f"params: {count_parameters(model)}")


## 2. Single-fold sanity check

In [ ]:
manifest = pd.read_csv(ROOT / "outputs" / "preprocessed" / "manifest.csv")
cache_root = ROOT / "outputs" / "preprocessed"

hold_out = manifest["subject_id"].unique()[0]
train_subj = [s for s in manifest["subject_id"].unique() if s != hold_out]
tr_ds = WindowDataset(manifest, cache_root, subjects=train_subj, augment=True)
va_ds = WindowDataset(manifest, cache_root, subjects=[hold_out], augment=False)

model = resnet18_finetune(in_channels=2, pretrained=True, freeze_until="layer3")
fold = train_one_fold(model, tr_ds, va_ds, epochs=5, batch_size=16,
                     lr=3e-4, device=device, verbose=True)
print(f"\nval window-AUC: {fold['best_val_window_auc']:.3f}")


## 3. Full LOSO (slow — strongly prefer GPU)

In [ ]:
EPOCHS = 10
RUN_FULL = False

if RUN_FULL:
    folds = loso_cv(
        model_factory=lambda: resnet18_finetune(in_channels=2, pretrained=True, freeze_until="layer3"),
        manifest=manifest, cache_root=cache_root,
        channel_mode="both", epochs=EPOCHS, lr=3e-4, batch_size=16, device=device,
    )
    print(loso_summary(folds))
    folds.to_csv(ROOT / "outputs" / "metrics" / "resnet_folds.csv", index=False)
else:
    print("set RUN_FULL = True")


### Notes on freezing schedule

`freeze_until="layer3"` leaves only `layer4` and `fc` trainable — recommended starting point given how few subjects we have. If subject-AUC plateaus low, unfreeze `layer3` next, then `layer2`. Avoid full fine-tune unless the cache grows substantially (more subjects, more augmentation).
